# 05 — Multivariate Analysis
## Guided examples with pandas, NumPy, Matplotlib, and Seaborn

This notebook creates a reproducible synthetic student-performance dataset. It demonstrates correlations, heatmaps, pair plots, grouped summaries, pivot tables, cross-tabulations, and group-specific relationships. These examples are illustrative and do not establish causation.

## 1. Imports and synthetic dataset
Each row represents one synthetic student. `student_id` is an identifier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)
n = 240
program = rng.choice(["Data Science", "Cybersecurity", "Software Engineering"], size=n, p=[0.36, 0.30, 0.34])
study_group = rng.choice(["Low", "Medium", "High"], size=n, p=[0.28, 0.46, 0.26])
study_hours = np.clip(rng.normal(8, 3, size=n), 0, 20)
attendance_pct = np.clip(rng.normal(82, 10, size=n), 45, 100)
prior_score = np.clip(rng.normal(68, 12, size=n), 25, 98)
program_effect = pd.Series(program).map({"Data Science": 2.5, "Cybersecurity": 0.0, "Software Engineering": 1.2}).to_numpy()
group_effect = pd.Series(study_group).map({"Low": -4, "Medium": 0, "High": 4}).to_numpy()
noise = rng.normal(0, 7, size=n)
exam_score = np.clip(18 + 1.6*study_hours + 0.22*attendance_pct + 0.32*prior_score + program_effect + group_effect + noise, 0, 100)
df = pd.DataFrame({
    "student_id": [f"S{i:03d}" for i in range(1, n+1)], "program": program,
    "study_group": study_group, "study_hours": study_hours.round(1),
    "attendance_pct": attendance_pct.round(1), "prior_score": prior_score.round(1),
    "exam_score": exam_score.round(1)
})
df["result_band"] = pd.cut(df["exam_score"], bins=[-0.1, 49, 69, 84, 100], labels=["Needs support", "Developing", "Proficient", "Advanced"])
missing_study = rng.choice(df.index, size=8, replace=False)
missing_prior = rng.choice(df.index, size=6, replace=False)
df.loc[missing_study, "study_hours"] = np.nan
df.loc[missing_prior, "prior_score"] = np.nan
df.head()


## 2. Structure, row grain, and quality checks

In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicate student IDs:", int(df["student_id"].duplicated().sum()))
display(df.sample(8, random_state=10))
quality = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing_count": df.isna().sum(), "missing_percent": (df.isna().mean()*100).round(2), "distinct_count": df.nunique(dropna=False)})
display(quality.sort_values("missing_percent", ascending=False))


## 3. Pearson and Spearman correlation matrices

In [ ]:
numeric = df.select_dtypes(include="number").drop(columns=["student_id"], errors="ignore")
pearson_corr = numeric.corr(method="pearson")
spearman_corr = numeric.corr(method="spearman")
display(pearson_corr.round(2))
display(spearman_corr.round(2))


## 4. Correlation heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0, square=True)
plt.title("Pearson correlation matrix — student performance")
plt.tight_layout()
plt.show()


## 5. Pair plot with program as hue

In [ ]:
pair_cols = ["study_hours", "attendance_pct", "prior_score", "exam_score", "program"]
sns.pairplot(df[pair_cols], hue="program", corner=True, plot_kws={"alpha": 0.65, "s": 25})
plt.show()


## 6. Scatter plot with multiple variables

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="study_hours", y="exam_score", hue="program", size="attendance_pct", sizes=(25, 150), alpha=0.75)
plt.title("Study hours and exam score, by program and attendance")
plt.xlabel("Study hours per week")
plt.ylabel("Exam score")
plt.tight_layout()
plt.show()


## 7. Grouped summaries and pivot table

In [ ]:
group_summary = df.groupby(["program", "study_group"], observed=False)["exam_score"].agg(count="count", mean="mean", median="median", std="std").round(2)
display(group_summary)
mean_pivot = df.pivot_table(values="exam_score", index="program", columns="study_group", aggfunc="mean", observed=False).round(1)
display(mean_pivot)


## 8. Categorical combinations: counts and row percentages

In [ ]:
counts = pd.crosstab(df["program"], df["result_band"], dropna=False)
row_percent = pd.crosstab(df["program"], df["result_band"], normalize="index", dropna=False).mul(100).round(1)
display(counts)
display(row_percent)


## 9. Group-specific correlations and facets

In [ ]:
rows = []
for program_name, group in df.groupby("program", observed=False):
    valid = group[["study_hours", "exam_score"]].dropna()
    rows.append({"program": program_name, "n_complete": len(valid), "pearson_r": valid["study_hours"].corr(valid["exam_score"])})
display(pd.DataFrame(rows).round(3))
g = sns.relplot(data=df, x="study_hours", y="exam_score", col="program", kind="scatter", height=3.5, facet_kws={"sharex": True, "sharey": True}, alpha=0.7)
g.set_axis_labels("Study hours per week", "Exam score")
g.set_titles("{col_name}")
plt.show()


## 10. Screen for strongly correlated pairs
A threshold is a review flag, not a universal rule or proof of multicollinearity.

In [ ]:
corr = numeric.corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().rename("correlation").reset_index()
pairs.columns = ["variable_1", "variable_2", "correlation"]
pairs["absolute_correlation"] = pairs["correlation"].abs()
display(pairs.sort_values("absolute_correlation", ascending=False))
display(pairs[pairs["absolute_correlation"] > 0.75])


## 11. Interpretation prompts

1. Which numerical variables show the strongest observed association?
2. Does the study-hours/exam-score pattern look similar across programs?
3. How do average scores differ across program and study-group combinations?
4. Are any category combinations represented by few students?
5. What missingness or measurement issues could affect the findings?
6. Name one plausible confounder and a follow-up analysis.
7. Which statements are descriptive, and which would require stronger evidence to be causal?

## Key takeaways
Use complementary views, inspect group sizes and missingness, and treat correlations as exploratory. The synthetic data is for learning only and should not be used to make claims about real students or programs.